# Homework 2 — Loops, functions and files

About 45-60 minutes. This time you work with a **real** Wavely export: all plays of 1 September 2026.

Columns: `event_id, event_ts, user_id, track_id, ms_played, device, country, skipped, shuffle`

In [1]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: C:\Users\LENOVO\Desktop\Creative Hub\MODUL 1\data-engineering-modul-1\data


### Exercise 1 — Load the file
Read `DATA / "raw" / "plays_2026-09-01.csv"` with `csv.DictReader` into a list called `rows`.

In [2]:
import csv

with open(DATA / "raw" / "plays_2026-09-01.csv", newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

print(len(rows), "rows")  # expected: 4334 rows
print(rows[0])

4334 rows
{'event_id': '7a28c2bc7c9e', 'event_ts': '2026-09-01 00:00:09', 'user_id': 'U0525', 'track_id': 'T0028', 'ms_played': '226000', 'device': 'desktop', 'country': 'AT', 'skipped': 'false', 'shuffle': 'false'}


Notice: **every value is a string**, even `"226000"` and `"false"`. CSV files have no types — remember this, it matters below.

### Exercise 2 — Plays per device
Build `plays_per_device`, a dict device → number of rows.

In [3]:
plays_per_device = {}
for row in rows:
    device = row["device"]
    plays_per_device[device] = plays_per_device.get(device, 0) + 1

print(plays_per_device)

{'desktop': 884, 'mobile': 2620, 'smart_speaker': 423, 'web': 407}


### Exercise 3 — Broken rows
A client bug sometimes sends `ms_played = -1`. Write `is_valid(row)` that returns `False` for those rows
(careful: the value is a string!). Count the invalid rows in `invalid_count`.

In [4]:
def is_valid(row):
    return int(row["ms_played"]) != -1


invalid_count = 0
for row in rows:
    if not is_valid(row):
        invalid_count += 1

print(invalid_count)  # expected: 14

14


### Exercise 4 — Duplicates
Our app sometimes sends the same event twice. Count how often every `event_id` appears, then compute
`duplicate_ids`: how many event ids appear more than once.

In [5]:
event_counts = {}
for row in rows:
    event_id = row["event_id"]
    event_counts[event_id] = event_counts.get(event_id, 0) + 1

duplicate_ids = 0
for count in event_counts.values():
    if count > 1:
        duplicate_ids += 1

print(duplicate_ids)  # expected: 64

64


### Exercise 5 — Listening hours per country
Build `hours_per_country`: country → total hours played, counting only rows that are valid **and** not skipped.
Round the values to 1 decimal at the end. (Hint: `row["skipped"] == "false"`.)

In [6]:
ms_per_country = {}
for row in rows:
    if is_valid(row) and row["skipped"] == "false":
        country = row["country"]
        ms_per_country[country] = ms_per_country.get(country, 0) + int(row["ms_played"])

# Convert milliseconds to hours and round only after summing.
hours_per_country = {}
for country, milliseconds in ms_per_country.items():
    hours_per_country[country] = round(milliseconds / 3_600_000, 1)

print(hours_per_country)

{'AT': 45.0, 'CZ': 24.3, 'DE': 32.9, 'SK': 28.2, 'HU': 18.2, 'PL': 18.2}


### Exercise 6 — The busiest hour
`event_ts` looks like `"2026-09-01 18:22:05"`. The hour is characters 11 and 12: `row["event_ts"][11:13]`.
Find `busiest_hour` (as a string like `"19"`) and how many plays it had (`busiest_count`).

In [7]:
plays_per_hour = {}
for row in rows:
    hour = row["event_ts"][11:13]
    plays_per_hour[hour] = plays_per_hour.get(hour, 0) + 1

busiest_hour = max(plays_per_hour, key=plays_per_hour.get)
busiest_count = plays_per_hour[busiest_hour]

print(busiest_hour, busiest_count)

20 360


### Exercise 7 (Stretch) — One function for any day
Write `summarize_day(path)` that returns a dict with `file` (just the file name), `rows`, `unique_events`,
`skip_rate` (share of rows with skipped `"true"`, rounded to 3 decimals) and `top_device`.

Run it on the first three files of `sorted((DATA / "raw").glob("plays_*.csv"))`.

In [8]:
def summarize_day(path):
    path = Path(path)
    with open(path, newline="", encoding="utf-8") as file:
        day_rows = list(csv.DictReader(file))

    unique_events = set()
    device_counts = {}
    skipped_count = 0
    for row in day_rows:
        unique_events.add(row["event_id"])
        device = row["device"]
        device_counts[device] = device_counts.get(device, 0) + 1
        if row["skipped"] == "true":
            skipped_count += 1

    return {
        "file": path.name,
        "rows": len(day_rows),
        "unique_events": len(unique_events),
        "skip_rate": round(skipped_count / len(day_rows), 3) if day_rows else 0.0,
        "top_device": max(device_counts, key=device_counts.get) if device_counts else None,
    }


for path in sorted((DATA / "raw").glob("plays_*.csv"))[:3]:
    print(summarize_day(path))

{'file': 'plays_2026-09-01.csv', 'rows': 4334, 'unique_events': 4270, 'skip_rate': 0.308, 'top_device': 'mobile'}
{'file': 'plays_2026-09-02.csv', 'rows': 3902, 'unique_events': 3845, 'skip_rate': 0.282, 'top_device': 'mobile'}
{'file': 'plays_2026-09-03.csv', 'rows': 4338, 'unique_events': 4274, 'skip_rate': 0.297, 'top_device': 'mobile'}
